# Rate Limits, Retries, Backoff, and Circuit Breakers

**Phase 06** · ~60 minutes · Python

**Prerequisites:** See lesson README

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thepandanlabs/applied-ai-from-scratch/blob/main/notebooks/phase-06/07-rate-limits-retries-circuit-breakers.ipynb)

---

> This notebook is auto-generated from the [Applied AI From Scratch](https://github.com/thepandanlabs/applied-ai-from-scratch) curriculum.  
> Run cells top to bottom. Set your API key in the **Setup** cell below.


In [ ]:
!pip install -q anthropic

import os

try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    pass  # Running locally — set ANTHROPIC_API_KEY before this cell

print("Setup complete. API key set:", bool(os.environ.get("ANTHROPIC_API_KEY")))

## The Problem

Your AI service calls the Anthropic API on every request. The API works perfectly in testing. In production, at 3 p.m. on a Tuesday, it starts returning HTTP 429 (Too Many Requests). Your service dutifully retries the failed call. All 200 instances of your service retry simultaneously. The API receives 200 requests at the exact same moment. It returns 429 to all of them. All 200 instances retry again at the same moment. You have just turned a temporary rate limit into a sustained denial-of-service attack against the API you depend on.

The second failure mode: your service has been getting 500 errors from the API for 10 minutes. Every incoming user request triggers a new attempt that fails i...

## The Concept

### Three Classes of API Failures

```
┌──────────────────────────────────────────────────────────────────┐
│  FAILURE CLASS        SIGNAL              CORRECT RESPONSE       │
├──────────────────────────────────────────────────────────────────┤
│  Transient error      500, 503, network   Retry with backoff     │
│                       timeout, DNS flap                          │
├──────────────────────────────────────────────────────────────────┤
│  Rate limit           429 + Retry-After   Wait for Retry-After,  │
│                       header              then retry             │
├────────...

_Read the full lesson narrative in `docs/en.md` or on the course site._

## Implementation

Run each cell in order:

### Setup & Imports

In [ ]:
"""
Rate Limits, Retries, Backoff, and Circuit Breakers for AI API calls.

Implements:
- exponential backoff with jitter (avoids thundering herd)
- Retry-After header parsing (respects API rate limit signals)
- circuit breaker (stops calling a broken dependency; fast-fails callers)
- ResilientClient wrapping the Anthropic SDK

Usage:
    ANTHROPIC_API_KEY=sk-... python main.py
"""

import os
import random
import sys
import threading
import time
from dataclasses import dataclass, field
from enum import Enum
from typing import Any, Callable

import anthropic

### Exponential backoff with jitter

In [ ]:
def backoff_with_jitter(
    attempt: int,
    base_delay: float = 1.0,
    max_delay: float = 60.0,
    jitter_factor: float = 0.5,
) -> float:
    """
    Compute a retry delay for the given attempt number (0-indexed).

    Formula: min(base * 2^attempt, max_delay) + uniform(0, jitter_factor * delay)

    Without jitter, all retrying service instances wake up at the same moment
    and create a thundering herd. Jitter spreads the wakeup times.

    attempt=0 -> ~1s, attempt=1 -> ~2s, attempt=2 -> ~4s, attempt=3 -> ~8s
    (all +/- up to 50% random jitter)
    """
    delay = min(base_delay * (2 ** attempt), max_delay)
    jitter = random.uniform(0, jitter_factor * delay)
    return delay + jitter


def parse_retry_after(headers: dict) -> float | None:
    """
    Parse the Retry-After header from a 429 response.

    Returns the number of seconds to wait, or None if the header is absent.
    When present, this value takes priority over backoff calculations:
    the API is telling you exactly how long to wait.
    """
    value = headers.get("retry-after") or headers.get("Retry-After")
    if value is None:
        return None
    try:
        return float(value)
    except (ValueError, TypeError):
        return None

### Circuit breaker

In [ ]:
class CircuitState(Enum):
    CLOSED = "closed"        # normal: calls pass through
    OPEN = "open"            # failing: calls rejected immediately (fast-fail)
    HALF_OPEN = "half_open"  # probing: one call allowed through to test recovery


class CircuitOpenError(Exception):
    """Raised when a call is rejected because the circuit breaker is OPEN."""
    pass


@dataclass
class CircuitBreaker:
    """
    Thread-safe circuit breaker for wrapping external API calls.

    States:
      CLOSED -> OPEN: after `failure_threshold` consecutive failures
      OPEN -> HALF_OPEN: after `recovery_timeout` seconds
      HALF_OPEN -> CLOSED: if the probe call succeeds
      HALF_OPEN -> OPEN: if the probe call fails

    In the OPEN state, calls are rejected immediately without reaching the API.
    This protects the API from a flood of failing requests and protects callers
    from long timeouts while the dependency is broken.
    """

    failure_threshold: int = 5
    recovery_timeout: float = 60.0  # seconds before attempting recovery

    _state: CircuitState = field(default=CircuitState.CLOSED, init=False)
    _failure_count: int = field(default=0, init=False)
    _last_failure_time: float = field(default=0.0, init=False)
    _lock: threading.Lock = field(default_factory=threading.Lock, init=False)

    def call(self, fn: Callable[[], Any]) -> Any:
        """
        Execute fn through the circuit breaker.
        Raises CircuitOpenError immediately if the circuit is OPEN.
        """
        with self._lock:
            current_state = self._compute_state()

        if current_state == CircuitState.OPEN:
            elapsed = time.time() - self._last_failure_time
            remaining = max(0.0, self.recovery_timeout - elapsed)
            raise CircuitOpenError(
                f"Circuit is OPEN (failures={self._failure_count}). "
                f"Recovery probe in {remaining:.1f}s."
            )

        try:
            result = fn()
            self._record_success()
            return result
        except Exception:
            self._record_failure()
            raise

    def _compute_state(self) -> CircuitState:
        """Transitions OPEN -> HALF_OPEN if recovery timeout has elapsed."""
        if (
            self._state == CircuitState.OPEN
            and time.time() - self._last_failure_time >= self.recovery_timeout
        ):
            self._state = CircuitState.HALF_OPEN
        return self._state

    def _record_success(self) -> None:
        with self._lock:
            self._failure_count = 0
            self._state = CircuitState.CLOSED

    def _record_failure(self) -> None:
        with self._lock:
            self._failure_count += 1
            self._last_failure_time = time.time()
            if (
                self._state == CircuitState.HALF_OPEN
                or self._failure_count >= self.failure_threshold
            ):
                self._state = CircuitState.OPEN

    @property
    def state(self) -> CircuitState:
        with self._lock:
            return self._compute_state()

    @property
    def failure_count(self) -> int:
        return self._failure_count

### ResilientClient

In [ ]:
class ResilientClient:
    """
    Anthropic client with retry logic and circuit breaker.

    Failure handling:
      - 429 RateLimitError: wait for Retry-After header (or backoff), retry
      - 5xx APIStatusError: retry with exponential backoff + jitter
      - Non-retryable 4xx (except 429): raise immediately, no retry
      - After failure_threshold failures: circuit opens, calls fail fast
      - CircuitOpenError: propagated to caller immediately
    """

    RETRYABLE_STATUS_CODES = {429, 500, 502, 503, 504}

    def __init__(
        self,
        api_key: str,
        max_attempts: int = 4,
        base_delay: float = 1.0,
        max_delay: float = 60.0,
        failure_threshold: int = 5,
        recovery_timeout: float = 60.0,
    ):
        self.client = anthropic.Anthropic(api_key=api_key)
        self.max_attempts = max_attempts
        self.base_delay = base_delay
        self.max_delay = max_delay
        self.circuit = CircuitBreaker(
            failure_threshold=failure_threshold,
            recovery_timeout=recovery_timeout,
        )

    def create_message(self, **kwargs) -> anthropic.types.Message:
        """
        Call client.messages.create with retry + circuit breaker.

        All kwargs are forwarded to the Anthropic messages.create call.
        Raises CircuitOpenError if the circuit is open (fast-fail).
        Raises the last error after max_attempts are exhausted.
        """
        last_error: Exception | None = None

        for attempt in range(self.max_attempts):
            try:
                return self.circuit.call(
                    lambda: self.client.messages.create(**kwargs)
                )

            except CircuitOpenError:
                # Fast-fail: do not retry when circuit is open
                raise

            except anthropic.RateLimitError as e:
                # 429: use Retry-After header if present; otherwise use backoff
                retry_after = None
                if hasattr(e, "response") and e.response is not None:
                    retry_after = parse_retry_after(dict(e.response.headers))

                wait = retry_after if retry_after is not None else backoff_with_jitter(
                    attempt, self.base_delay, self.max_delay
                )
                print(
                    f"[attempt {attempt+1}/{self.max_attempts}] Rate limited (429). "
                    f"Waiting {wait:.1f}s "
                    f"({'Retry-After header' if retry_after else 'backoff'})",
                    file=sys.stderr,
                )
                if attempt < self.max_attempts - 1:
                    time.sleep(wait)
                last_error = e

            except anthropic.APIStatusError as e:
                status = getattr(e, "status_code", 0)
                if status in self.RETRYABLE_STATUS_CODES:
                    wait = backoff_with_jitter(attempt, self.base_delay, self.max_delay)
                    print(
                        f"[attempt {attempt+1}/{self.max_attempts}] API error {status}. "
                        f"Retrying in {wait:.1f}s",
                        file=sys.stderr,
                    )
                    if attempt < self.max_attempts - 1:
                        time.sleep(wait)
                    last_error = e
                else:
                    # Non-retryable: 400, 401, 403, 404, etc.
                    raise

            except (anthropic.APIConnectionError, anthropic.APITimeoutError) as e:
                wait = backoff_with_jitter(attempt, self.base_delay, self.max_delay)
                print(
                    f"[attempt {attempt+1}/{self.max_attempts}] Connection error: {e}. "
                    f"Retrying in {wait:.1f}s",
                    file=sys.stderr,
                )
                if attempt < self.max_attempts - 1:
                    time.sleep(wait)
                last_error = e

        raise last_error or RuntimeError(
            f"All {self.max_attempts} retry attempts exhausted"
        )

### Demo

In [ ]:
def main():
    api_key = os.environ.get("ANTHROPIC_API_KEY")
    if not api_key:
        print("Set ANTHROPIC_API_KEY to run the demo", file=sys.stderr)
        sys.exit(1)

    client = ResilientClient(
        api_key=api_key,
        max_attempts=4,
        base_delay=1.0,
        failure_threshold=3,
        recovery_timeout=30.0,
    )

    print(f"Circuit state: {client.circuit.state.value}")

    msg = client.create_message(
        model="claude-3-5-haiku-20241022",
        max_tokens=256,
        messages=[{"role": "user", "content": "Explain circuit breakers in one sentence."}],
    )
    print(f"Response: {msg.content[0].text}")
    print(f"Circuit state after success: {client.circuit.state.value}")
    print(f"Failures: {client.circuit.failure_count}")

### Demo

In [ ]:
main()

---

## Self-Check

Answer these without running code first:

**1. What problem occurs when the API recovers, and how does jitter fix it?**

- A. No problem. A fixed 2-second delay is appropriate and all instances will reconnect smoothly.
- B. All 150 instances wake up simultaneously at t=2s, t=4s, t=6s and send synchronized retry bursts. This thundering herd can immediately overwhelm the recovering API with 150 simultaneous requests per wave. Jitter adds random variation to each delay, spreading 150 instances across a 1-3 second window instead of all firing at the same moment.
- C. The fixed delay causes retries to fail because the API only accepts requests within a 100ms window after recovery.
- D. Jitter is only needed when retry intervals exceed 60 seconds. For 2-second delays, it has no meaningful effect.

**2. How long should your service wait before retrying?**

- A. 3 seconds, because the backoff formula is more conservative than the Retry-After header.
- B. 8 seconds, because the Retry-After header is the API's explicit instruction about when it will accept requests again.
- C. 11 seconds (3 + 8), to be safe by combining both values.
- D. 0 seconds. Retry-After headers are advisory and can be ignored if your service needs low latency.

**3. What happens to that request, and why is this behavior beneficial?**

- A. The circuit attempts the API call and waits for a response, just as it would in CLOSED state.
- B. The request is rejected immediately with CircuitOpenError without calling the API. This is beneficial because it returns a fast error to the user in microseconds instead of making them wait for a 5-second timeout on a broken API.
- C. The circuit transitions to HALF_OPEN and allows the request through as a probe, since 10 seconds have passed.
- D. The circuit waits 50 more seconds (60 - 10 = 50) and then processes the request.

**4. What is the correct behavior for a 400 error, and why?**

- A. Retry all 4 times, because the prompt might fit within the context window on a subsequent attempt.
- B. Do not retry. A 400 error is a client error -- the request itself is malformed. Retrying the identical request will always return 400. Retrying wastes time and quota.
- C. Retry once with a longer delay, then give up. The API might accept the request on a second attempt if its context tracking resets.
- D. Retry with a smaller max_tokens value to reduce the request size.

**5. What happens to those 50 users' requests during the 2-hour outage?**

- A. The requests queue up and complete successfully when the API recovers.
- B. Each request ties up a thread for up to 50 seconds (10 attempts * 5 seconds each). With 50 concurrent requests, 50 threads are blocked. The service's thread pool is exhausted and new incoming requests are rejected.
- C. The service automatically reduces max_attempts during high load to protect its thread pool.
- D. After the first failed request, the service automatically pauses all retries until the API recovers.

**6. What state is the circuit in after those 3 failures in HALF_OPEN?**

- A. CLOSED. The one successful probe proved the API recovered, and the subsequent failures are treated as new transient errors.
- B. OPEN. The first failure in HALF_OPEN returns the circuit to OPEN immediately, resetting the recovery timeout.
- C. HALF_OPEN. The circuit stays in HALF_OPEN until failure_threshold failures accumulate.
- D. A new DEGRADED state that allows 50% of requests through.

**7. What specific failure scenario does the circuit breaker handle that tenacity alone cannot?**

- A. tenacity cannot handle 429 errors. Only a circuit breaker can parse Retry-After headers.
- B. When the API is down for minutes, tenacity will exhaust its 4 attempts per request and raise an exception. But each new incoming user request starts a fresh set of 4 attempts, each waiting up to 60 seconds. A circuit breaker stops all calls immediately after N failures, preventing new requests from entering the retry loop at all.
- C. tenacity only works with synchronous code. Circuit breakers handle async code.
- D. tenacity does not support jitter. The circuit breaker's backoff formula includes jitter by default.

**8. Will the circuit breaker open in this scenario, and is that the correct behavior?**

- A. Yes, the circuit will open after 5 failures. This is correct because it stops the service from sending broken requests.
- B. No, the circuit should not open for 400 errors because 400 is a client error that the client caused, not an API health issue. Opening the circuit would mask a bug that needs to be fixed.
- C. The circuit will open and then immediately close when the probe succeeds, creating a loop.
- D. 400 errors are not counted by circuit breakers because they are defined as client errors, not server errors.

_Answers are in `checks.json` in the lesson directory._